# Server Prototype: FastAPI + ngrok Whisper STT Server

**목적**: 모바일 앱에서 호출 가능한 REST API 형태의 STT 서버 프로토타입을 구축합니다.

**시도한 내용**:
- FastAPI 기반 `/transcribe` 엔드포인트 구현 (오디오 파일 업로드 → 텍스트 반환)
- ngrok 터널링으로 Colab 환경에서 외부 접근 가능한 퍼블릭 URL 생성

**결론**: 서버 기반 추론의 기본 아키텍처(앱 → API → 모델 추론 → 응답)를 검증. 최종 서비스의 GCP Cloud Run 서버 설계의 기초가 됨.


In [1]:
!pip install -U transformers python-multipart fastapi pyngrok uvicorn

In [5]:
import librosa
from fastapi import FastAPI, UploadFile, File, Request, Form
from starlette.datastructures import UploadFile as StarletteUploadFile
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
from io import BytesIO

app = FastAPI()

# Load fine-tuned Whisper model
model_path = "/content/drive/MyDrive/JBNU/2024-2/Capstone/Whisper/Telos_LLM"  # Replace with your model path
processor_path = "/content/drive/MyDrive/JBNU/2024-2/Capstone/Whisper/Telos_LLM_Processor"  # Replace with your model path
model = WhisperForConditionalGeneration.from_pretrained(model_path)
processor = WhisperProcessor.from_pretrained(processor_path)



@app.post("/transcribe")
async def transcribe_audio(file: UploadFile = File(...)):
    # print('file received')
    # Load audio file
    audio_bytes = await file.read()

    # Load audio using librosa
    audio, sample_rate = librosa.load(BytesIO(audio_bytes), sr=16000)  # Ensure 16kHz sample rate

    # Convert audio to NumPy array and extract features
    input_features = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features

    # Generate transcription
    generated_ids = model.generate(input_features)
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {"transcription": transcription}

@app.get('/')
async def root():
    return {'hello': 'world'}

In [6]:
import os

import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
auth_token = os.getenv("NGROK_AUTH_TOKEN")

# Set the authtoken
ngrok.set_auth_token(auth_token)

# Connect to ngrok
ngrok_tunnel = ngrok.connect(8000)

# Print the public URL
print('Public URL:', ngrok_tunnel.public_url)

# Apply nest_asyncio
nest_asyncio.apply()

# Run the uvicorn server
uvicorn.run(app, port=8000)

Public URL: https://0cee-34-125-209-118.ngrok-free.app


INFO:     Started server process [2820]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


file received


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     221.142.139.206:0 - "POST /transcribe HTTP/1.1" 200 OK


file received
INFO:     221.142.139.206:0 - "POST /transcribe HTTP/1.1" 200 OK


file received
INFO:     221.142.139.206:0 - "POST /transcribe HTTP/1.1" 200 OK


file received
INFO:     221.142.139.206:0 - "POST /transcribe HTTP/1.1" 200 OK


file received
INFO:     221.142.139.206:0 - "POST /transcribe HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [2820]
